# NeuroScope AI - Notebook 16: Report Generator

Generates structured radiology reports, PDFs, and patient summaries.

**What this notebook builds:**
1. Structured reports -- BI-RADS (breast), Lung-RADS (lung), LI-RADS (liver), brain MRI
2. PDF report generation -- professional clinical report layout
3. Patient-friendly plain language summaries (6th grade reading level)
4. Multi-language support -- English, Spanish, French, Hindi
5. ReportAgent -- full Agent 6 implementation replacing NB11 stub

---

## Cell 1 - Imports & Config

In [1]:
import os, sys, json, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ReportLab for PDF
try:
    from reportlab.lib.pagesizes import letter, A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.colors import HexColor, black, white, grey
    from reportlab.lib.units import inch, cm
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        HRFlowable, Image as RLImage, PageBreak
    )
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    REPORTLAB_AVAILABLE = True
    print('ReportLab : OK')
except ImportError:
    REPORTLAB_AVAILABLE = False
    print('ReportLab not installed -- run: pip install reportlab')
    print('Text reports will still work')

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
OUT  = os.path.join(BASE, 'outputs', 'nb16_reports')
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, BASE)

print(f'OUT : {OUT}')
print('Imports OK')

ReportLab : OK
OUT : C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports
Imports OK


---
## Cell 2 - Structured Report Templates

In [2]:
from datetime import datetime


# BI-RADS scoring definitions
BIRADS = {
    0: {'label': 'Incomplete',         'action': 'Additional imaging needed',         'cancer_risk': 'N/A'},
    1: {'label': 'Negative',           'action': 'Routine screening',                'cancer_risk': '0%'},
    2: {'label': 'Benign',             'action': 'Routine screening',                'cancer_risk': '0%'},
    3: {'label': 'Probably Benign',    'action': '6-month follow-up',                'cancer_risk': '<2%'},
    4: {'label': 'Suspicious',         'action': 'Tissue sampling recommended',      'cancer_risk': '2-95%'},
    5: {'label': 'Highly Suspicious',  'action': 'Biopsy strongly recommended',      'cancer_risk': '>95%'},
    6: {'label': 'Known Malignancy',   'action': 'Treatment planning',               'cancer_risk': 'N/A'},
}

# Lung-RADS scoring definitions
LUNGRADS = {
    1: {'label': 'Negative',        'action': 'Annual LDCT',              'cancer_risk': '<1%'},
    2: {'label': 'Benign Appearance','action': 'Annual LDCT',              'cancer_risk': '<1%'},
    3: {'label': 'Probably Benign', 'action': '6-month LDCT',             'cancer_risk': '1-2%'},
    4: {'label': 'Suspicious',
        'sub': {
            'A': {'action': '3-month LDCT or PET/CT',  'cancer_risk': '5-15%'},
            'B': {'action': 'Tissue sampling',         'cancer_risk': '>15%'},
            'X': {'action': 'Additional workup',       'cancer_risk': 'Variable'},
        }
    },
}

# LI-RADS scoring definitions
LIRADS = {
    1: {'label': 'Definitely benign',   'action': 'Continue surveillance',     'hcc_risk': '0%'},
    2: {'label': 'Probably benign',     'action': 'Continue surveillance',     'hcc_risk': '<10%'},
    3: {'label': 'Intermediate',        'action': '6-month MRI',               'hcc_risk': '10-50%'},
    4: {'label': 'Probably HCC',        'action': 'MDT review',                'hcc_risk': '>50%'},
    5: {'label': 'Definitely HCC',      'action': 'HCC treatment -- no biopsy needed', 'hcc_risk': '>95%'},
    'M': {'label': 'Malignant not HCC', 'action': 'Biopsy recommended',        'hcc_risk': 'N/A'},
    'TIV': {'label': 'Tumor in vein',   'action': 'Urgent MDT',                'hcc_risk': 'High'},
}


def assign_birads(cancer_prob, tumor_type='unknown'):
    """Auto-assign BI-RADS category from model output."""
    if cancer_prob < 0.05: return 1
    if cancer_prob < 0.10: return 2
    if cancer_prob < 0.25: return 3
    if cancer_prob < 0.75: return 4
    if cancer_prob < 0.95: return 5
    return 5


def assign_lungrads(nodule_prob, nodule_size_mm=None):
    """Auto-assign Lung-RADS from nodule detection probability."""
    if nodule_prob < 0.1: return '1'
    if nodule_prob < 0.2: return '2'
    if nodule_prob < 0.5: return '3'
    if nodule_prob < 0.8: return '4A'
    return '4B'


def generate_structured_report(scan_data):
    """
    Generate structured text report based on cancer type.
    scan_data: dict with cancer_type, verdict, cancer_prob, tumor_type,
               confidence, who_grade, recommendations, scan_id, patient info
    """
    ct          = scan_data.get('cancer_type', 'unknown')
    verdict     = scan_data.get('verdict', 'REVIEW_REQUIRED')
    prob        = scan_data.get('cancer_prob', 0.5)
    tumor_type  = scan_data.get('tumor_type', 'not classified')
    confidence  = scan_data.get('confidence', 0.0)
    who_grade   = scan_data.get('who_grade', '')
    recs        = scan_data.get('recommendations', [])
    scan_id     = scan_data.get('scan_id', 'N/A')
    patient_age = scan_data.get('patient_age', 'unknown')
    patient_sex = scan_data.get('patient_sex', 'unknown')
    now         = datetime.now().strftime('%Y-%m-%d %H:%M')

    # Standard header
    lines = [
        '=' * 70,
        f'  NEUROSCOPE AI RADIOLOGY REPORT',
        f'  Generated  : {now}',
        f'  Scan ID    : {scan_id}',
        f'  Cancer type: {ct.upper()}',
        f'  Patient    : Age {patient_age}, Sex {patient_sex}',
        '=' * 70,
        '',
    ]

    # Cancer-specific scoring
    if ct == 'breast':
        birads_cat = assign_birads(prob)
        birads_def = BIRADS[birads_cat]
        lines += [
            'BI-RADS ASSESSMENT',
            f'  Category    : BI-RADS {birads_cat} -- {birads_def["label"]}',
            f'  Cancer Risk : {birads_def["cancer_risk"]}',
            f'  Action      : {birads_def["action"]}',
            '',
        ]

    elif ct == 'lung':
        lr_cat = assign_lungrads(prob)
        lines += [
            'LUNG-RADS ASSESSMENT',
            f'  Category    : Lung-RADS {lr_cat}',
            f'  Action      : {LUNGRADS.get(int(lr_cat[0]), {}).get("action", "MDT review")}',
            '',
        ]

    elif ct == 'liver':
        lirads_cat = 5 if prob > 0.85 else 4 if prob > 0.5 else 3
        lirads_def = LIRADS[lirads_cat]
        lines += [
            'LI-RADS ASSESSMENT',
            f'  Category    : LI-RADS {lirads_cat} -- {lirads_def["label"]}',
            f'  HCC Risk    : {lirads_def["hcc_risk"]}',
            f'  Action      : {lirads_def["action"]}',
            '',
        ]

    # AI findings
    lines += [
        'AI ANALYSIS FINDINGS',
        f'  Verdict       : {verdict}',
        f'  Cancer Prob   : {prob:.1%}',
        f'  Tumor Type    : {tumor_type} (confidence: {confidence:.1%})',
    ]
    if who_grade:
        lines.append(f'  WHO Grade     : {who_grade}')
    lines.append('')

    # Treatment recommendations
    if recs:
        lines.append('TREATMENT RECOMMENDATIONS')
        for i, rec in enumerate(recs[:6], 1):
            rec_text = rec if isinstance(rec, str) else rec.get('rec', str(rec))
            lines.append(f'  {i}. {rec_text}')
        lines.append('')

    # Disclaimer
    lines += [
        'DISCLAIMER',
        '  AI-ASSISTED ANALYSIS. CLINICIAN REVIEW REQUIRED.',
        '  This report does not constitute a clinical diagnosis.',
        '  All findings must be verified by a qualified radiologist.',
        '  NeuroScope AI v1.0 -- Research Platform (not FDA cleared)',
        '=' * 70,
    ]

    return '\n'.join(lines)


# Demo
demo_breast = {
    'cancer_type'   : 'breast',
    'verdict'       : 'CANCER_FLAGGED',
    'cancer_prob'   : 0.9986,
    'tumor_type'    : 'malignant',
    'confidence'    : 0.9986,
    'scan_id'       : '20ce249f',
    'patient_age'   : 54,
    'patient_sex'   : 'F',
    'recommendations': [
        'Lumpectomy + RT equivalent to mastectomy (Stage I-II)',
        'Sentinel lymph node biopsy',
        'HER2/ER/PR testing mandatory',
    ]
}

report_text = generate_structured_report(demo_breast)
print(report_text)
print('Structured report templates OK')

  NEUROSCOPE AI RADIOLOGY REPORT
  Generated  : 2026-06-02 23:35
  Scan ID    : 20ce249f
  Cancer type: BREAST
  Patient    : Age 54, Sex F

BI-RADS ASSESSMENT
  Category    : BI-RADS 5 -- Highly Suspicious
  Cancer Risk : >95%
  Action      : Biopsy strongly recommended

AI ANALYSIS FINDINGS
  Verdict       : CANCER_FLAGGED
  Cancer Prob   : 99.9%
  Tumor Type    : malignant (confidence: 99.9%)

TREATMENT RECOMMENDATIONS
  1. Lumpectomy + RT equivalent to mastectomy (Stage I-II)
  2. Sentinel lymph node biopsy
  3. HER2/ER/PR testing mandatory

DISCLAIMER
  AI-ASSISTED ANALYSIS. CLINICIAN REVIEW REQUIRED.
  This report does not constitute a clinical diagnosis.
  All findings must be verified by a qualified radiologist.
  NeuroScope AI v1.0 -- Research Platform (not FDA cleared)
Structured report templates OK


---
## Cell 3 - PDF Report Generator

In [3]:
import os
from datetime import datetime

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
OUT  = os.path.join(BASE, 'outputs', 'nb16_reports')


def generate_pdf_report(scan_data, output_path, gradcam_path=None):
    """
    Generate professional PDF radiology report using ReportLab.
    Includes structured findings, BI-RADS/Lung-RADS/LI-RADS scoring,
    treatment recommendations, and optional Grad-CAM overlay image.
    """
    if not REPORTLAB_AVAILABLE:
        # Text fallback
        txt = generate_structured_report(scan_data)
        txt_path = output_path.replace('.pdf', '.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(txt)
        print(f'ReportLab not available -- text report saved: {txt_path}')
        return txt_path

    ct          = scan_data.get('cancer_type', 'unknown').upper()
    verdict     = scan_data.get('verdict', 'REVIEW_REQUIRED')
    prob        = scan_data.get('cancer_prob', 0.5)
    tumor_type  = scan_data.get('tumor_type', 'not classified')
    confidence  = scan_data.get('confidence', 0.0)
    who_grade   = scan_data.get('who_grade', '')
    recs        = scan_data.get('recommendations', [])
    scan_id     = scan_data.get('scan_id', 'N/A')
    patient_age = scan_data.get('patient_age', 'unknown')
    patient_sex = scan_data.get('patient_sex', 'unknown')

    # Colors
    NAVY    = HexColor('#1a2a4a')
    RED     = HexColor('#c0392b')
    GREEN   = HexColor('#1D9E75')
    AMBER   = HexColor('#EF9F27')
    LGRAY   = HexColor('#f5f5f5')
    MGRAY   = HexColor('#888888')

    verdict_color = {'CANCER_FLAGGED': RED, 'NORMAL': GREEN}.get(verdict, AMBER)

    doc    = SimpleDocTemplate(output_path, pagesize=A4,
                                topMargin=1.5*cm, bottomMargin=1.5*cm,
                                leftMargin=2*cm, rightMargin=2*cm)
    styles = getSampleStyleSheet()
    story  = []

    # Header
    title_style = ParagraphStyle('title', fontSize=16, textColor=NAVY,
                                  fontName='Helvetica-Bold', spaceAfter=4)
    sub_style   = ParagraphStyle('sub',   fontSize=10, textColor=MGRAY,
                                  spaceAfter=2)
    body_style  = ParagraphStyle('body',  fontSize=9,  spaceAfter=3)
    label_style = ParagraphStyle('label', fontSize=9,  fontName='Helvetica-Bold',
                                  textColor=NAVY)
    warn_style  = ParagraphStyle('warn',  fontSize=8,  textColor=MGRAY,
                                  fontName='Helvetica-Oblique')

    story.append(Paragraph('NeuroScope AI Radiology Report', title_style))
    story.append(HRFlowable(width='100%', thickness=2, color=NAVY))
    story.append(Spacer(1, 0.2*cm))

    # Metadata table
    meta = [
        ['Generated', datetime.now().strftime('%Y-%m-%d %H:%M'),
         'Scan ID', scan_id],
        ['Cancer Type', ct,
         'Patient', f'Age {patient_age}, {patient_sex}'],
        ['Software', 'NeuroScope AI v1.0',
         'Status', 'RESEARCH PLATFORM'],
    ]
    meta_table = Table(meta, colWidths=[3*cm, 6*cm, 3*cm, 6*cm])
    meta_table.setStyle(TableStyle([
        ('FONTNAME',  (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTNAME',  (2,0), (2,-1), 'Helvetica-Bold'),
        ('FONTSIZE',  (0,0), (-1,-1), 8),
        ('TEXTCOLOR', (0,0), (0,-1), NAVY),
        ('TEXTCOLOR', (2,0), (2,-1), NAVY),
        ('BACKGROUND',(0,0), (-1,-1), LGRAY),
        ('GRID',      (0,0), (-1,-1), 0.5, HexColor('#dddddd')),
        ('PADDING',   (0,0), (-1,-1), 4),
    ]))
    story.append(meta_table)
    story.append(Spacer(1, 0.3*cm))

    # Verdict banner
    verdict_table = Table([[Paragraph(
        f'<b>AI VERDICT: {verdict}</b>  |  Cancer Probability: {prob:.1%}  '
        f'|  Tumor Type: {tumor_type}  |  Confidence: {confidence:.1%}',
        ParagraphStyle('verd', fontSize=10, textColor=white, fontName='Helvetica-Bold')
    )]], colWidths=[17*cm])
    verdict_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), verdict_color),
        ('PADDING',    (0,0), (-1,-1), 8),
        ('ROUNDEDCORNERS', [4]),
    ]))
    story.append(verdict_table)
    story.append(Spacer(1, 0.3*cm))

    # Cancer-specific scoring
    cancer_type_lower = scan_data.get('cancer_type', 'unknown')
    if cancer_type_lower == 'breast':
        birads_cat = assign_birads(prob)
        birads_def = BIRADS[birads_cat]
        story.append(Paragraph('BI-RADS Assessment', label_style))
        birads_data = [
            ['Category', f'BI-RADS {birads_cat} -- {birads_def["label"]}'],
            ['Cancer Risk', birads_def['cancer_risk']],
            ['Recommended Action', birads_def['action']],
        ]
        bt = Table(birads_data, colWidths=[5*cm, 12*cm])
        bt.setStyle(TableStyle([
            ('FONTNAME',   (0,0), (0,-1), 'Helvetica-Bold'),
            ('FONTSIZE',   (0,0), (-1,-1), 9),
            ('BACKGROUND', (0,0), (0,-1), LGRAY),
            ('GRID',       (0,0), (-1,-1), 0.5, HexColor('#dddddd')),
            ('PADDING',    (0,0), (-1,-1), 4),
        ]))
        story.append(bt)
        story.append(Spacer(1, 0.2*cm))

    # Grad-CAM image if available
    if gradcam_path and os.path.exists(gradcam_path):
        story.append(Paragraph('AI Attention Map (Grad-CAM)', label_style))
        story.append(RLImage(gradcam_path, width=8*cm, height=4*cm))
        story.append(Spacer(1, 0.2*cm))

    # Treatment recommendations
    if recs:
        story.append(Paragraph('Treatment Recommendations', label_style))
        for i, rec in enumerate(recs[:6], 1):
            rec_text = rec if isinstance(rec, str) else rec.get('rec', str(rec))
            story.append(Paragraph(f'{i}. {rec_text}', body_style))
        story.append(Spacer(1, 0.2*cm))

    # WHO grade
    if who_grade:
        story.append(Paragraph(f'WHO Grade: {who_grade} (imaging-based estimate)', body_style))

    # Disclaimer
    story.append(Spacer(1, 0.5*cm))
    story.append(HRFlowable(width='100%', thickness=1, color=MGRAY))
    story.append(Paragraph(
        'AI-ASSISTED ANALYSIS. CLINICIAN REVIEW REQUIRED. '
        'This report does not constitute a clinical diagnosis. '
        'All findings must be verified by a qualified radiologist. '
        'NeuroScope AI v1.0 -- Research platform. Not FDA cleared.',
        warn_style
    ))

    doc.build(story)
    size_kb = os.path.getsize(output_path) / 1024
    print(f'PDF saved: {output_path} ({size_kb:.0f} KB)')
    return output_path


# Generate demo PDF reports
test_scans = [
    {
        'cancer_type': 'brain',  'verdict': 'CANCER_FLAGGED',
        'cancer_prob': 0.9989,   'tumor_type': 'glioma',
        'confidence' : 0.9907,   'who_grade': 'Grade IV (GBM)',
        'scan_id'    : '48a808fc','patient_age': 45, 'patient_sex': 'M',
        'recommendations': [
            'Maximal safe surgical resection',
            'Concurrent RT (60Gy) + temozolomide',
            'IDH1/2 + MGMT + 1p/19q molecular testing',
        ]
    },
    {
        'cancer_type': 'breast', 'verdict': 'CANCER_FLAGGED',
        'cancer_prob': 0.9986,   'tumor_type': 'malignant',
        'confidence' : 0.9986,
        'scan_id'    : '20ce249f','patient_age': 54, 'patient_sex': 'F',
        'recommendations': [
            'Lumpectomy + RT or mastectomy',
            'Sentinel lymph node biopsy',
            'HER2/ER/PR testing mandatory',
        ]
    },
    {
        'cancer_type': 'skin',   'verdict': 'NORMAL',
        'cancer_prob': 0.0078,   'tumor_type': 'nv',
        'confidence' : 0.9922,
        'scan_id'    : '841f187a','patient_age': 62, 'patient_sex': 'M',
        'recommendations': ['Routine dermoscopy follow-up in 12 months']
    },
]

generated = []
gradcam_p = os.path.join(BASE, 'outputs', 'nb14_explainability', 'gradcam_results.png')

for scan in test_scans:
    pdf_path = os.path.join(OUT, f'report_{scan["scan_id"]}_{scan["cancer_type"]}.pdf')
    out_path = generate_pdf_report(
        scan, pdf_path,
        gradcam_path=gradcam_p if scan['cancer_type'] in ('brain', 'skin') else None
    )
    generated.append(out_path)

print(f'\nGenerated {len(generated)} reports')

PDF saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports\report_48a808fc_brain.pdf (2021 KB)
PDF saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports\report_20ce249f_breast.pdf (3 KB)
PDF saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports\report_841f187a_skin.pdf (2021 KB)

Generated 3 reports


---
## Cell 4 - Patient-Friendly Plain Language Summaries

In [4]:
import os, json

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'


# Plain language templates (6th grade reading level)
# Deliberately simple -- patients should understand without medical knowledge
PLAIN_TEMPLATES = {
    'CANCER_FLAGGED': {
        'brain': (
            'Your brain scan shows something that needs attention from your doctor. '
            'The AI found an area that looks unusual. '
            'This does not mean you definitely have a serious illness -- '
            'your doctor will explain the results and next steps. '
            'Please contact your healthcare team as soon as possible.'
        ),
        'breast': (
            'Your mammogram shows an area that your doctor needs to look at more closely. '
            'Finding something on a mammogram is common and does not always mean cancer. '
            'Your doctor will likely recommend more tests to find out what it is. '
            'Please call your doctor\'s office to discuss the next steps.'
        ),
        'skin': (
            'The AI found features in your skin lesion that a doctor should check. '
            'This does not mean you have skin cancer. '
            'A dermatologist can examine it more closely and advise you. '
            'Please make an appointment with your doctor soon.'
        ),
        'lung': (
            'Your lung scan shows a spot that needs follow-up. '
            'Lung spots are often harmless and can come from old infections or scars. '
            'Your doctor will decide if you need another scan or more tests. '
            'Please contact your doctor to discuss the findings.'
        ),
        'liver': (
            'Your liver scan shows something your doctor needs to review. '
            'Your doctor will tell you what tests come next. '
            'Please contact your healthcare team promptly.'
        ),
        'spine': (
            'Your spine scan shows some changes that may be causing your symptoms. '
            'These changes are common and can often be treated with physical therapy or other options. '
            'Your doctor will explain the best treatment plan for you.'
        ),
        'default': (
            'Your scan shows findings that your doctor needs to review with you. '
            'Please contact your healthcare provider to discuss the results and next steps.'
        ),
    },
    'NORMAL': {
        'default': (
            'The AI analysis of your scan did not find any concerning areas. '
            'This is good news. However, AI analysis is a tool to help doctors -- '
            'not a final result. Your doctor will review the scan and confirm the findings. '
            'Continue your regular health check-ups as recommended by your doctor.'
        )
    },
    'REVIEW_REQUIRED': {
        'default': (
            'The AI could not give a clear result for your scan and a specialist needs to review it. '
            'This happens sometimes and does not mean something is wrong. '
            'Your doctor will look at the scan and explain the findings to you.'
        )
    }
}


# Multi-language translations (key phrases)
TRANSLATIONS = {
    'please_contact': {
        'en': 'Please contact your doctor.',
        'es': 'Por favor, contacte a su médico.',
        'fr': 'Veuillez contacter votre médecin.',
        'hi': 'कृपया अपने डॉक्टर से संपर्क करें।',
        'ar': 'يرجى التواصل مع طبيبك.',
        'zh': '请联系您的医生。',
    },
    'ai_assisted': {
        'en': 'This report was created with AI assistance. Your doctor will review it.',
        'es': 'Este informe fue creado con asistencia de IA. Su médico lo revisará.',
        'fr': 'Ce rapport a été créé avec l\'aide de l\'IA. Votre médecin le vérifiera.',
        'hi': 'यह रिपोर्ट AI की मदद से बनाई गई है। आपके डॉक्टर इसकी समीक्षा करेंगे।',
        'ar': 'تم إنشاء هذا التقرير بمساعدة الذكاء الاصطناعي. سيراجعه طبيبك.',
        'zh': '本报告由AI辅助生成。您的医生将进行审查。',
    },
}


def generate_patient_summary(scan_data, language='en'):
    """
    Generate patient-friendly plain language summary.
    Returns: dict with 'summary' text and 'language' tag
    """
    verdict     = scan_data.get('verdict', 'REVIEW_REQUIRED')
    cancer_type = scan_data.get('cancer_type', 'default')

    templates = PLAIN_TEMPLATES.get(verdict, PLAIN_TEMPLATES['REVIEW_REQUIRED'])
    summary   = templates.get(cancer_type) or templates.get('default', '')

    # Append multi-language footer
    footer = TRANSLATIONS['ai_assisted'].get(language,
             TRANSLATIONS['ai_assisted']['en'])
    contact= TRANSLATIONS['please_contact'].get(language,
             TRANSLATIONS['please_contact']['en'])

    full_text = f'{summary}\n\n{contact}\n{footer}'

    return {'summary': full_text, 'language': language, 'verdict': verdict}


# Demo patient summaries
print('Patient summary examples:')
print()
for scan in test_scans:
    summary = generate_patient_summary(scan)
    print(f'{scan["cancer_type"].upper()} ({scan["verdict"]}):')
    print(f'  {summary["summary"][:200]}...')
    print()

# Multi-language demo
print('Multi-language support:')
for lang, name in [('en','English'),('es','Spanish'),('fr','French'),('hi','Hindi')]:
    s = generate_patient_summary(test_scans[0], language=lang)
    lines = s['summary'].split('\n')
    print(f'  {name:10s}: {lines[-1]}')

print('\nPatient summaries OK')

Patient summary examples:

BRAIN (CANCER_FLAGGED):
  Your brain scan shows something that needs attention from your doctor. The AI found an area that looks unusual. This does not mean you definitely have a serious illness -- your doctor will explain the...

BREAST (CANCER_FLAGGED):
  Your mammogram shows an area that your doctor needs to look at more closely. Finding something on a mammogram is common and does not always mean cancer. Your doctor will likely recommend more tests to...

SKIN (NORMAL):
  The AI analysis of your scan did not find any concerning areas. This is good news. However, AI analysis is a tool to help doctors -- not a final result. Your doctor will review the scan and confirm th...

Multi-language support:
  English   : This report was created with AI assistance. Your doctor will review it.
  Spanish   : Este informe fue creado con asistencia de IA. Su médico lo revisará.
  French    : Ce rapport a été créé avec l'aide de l'IA. Votre médecin le vérifiera.
  Hindi   

---
## Cell 5 - Full ReportAgent (replaces NB11 stub)

In [5]:
import os, json
from datetime import datetime

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
OUT  = os.path.join(BASE, 'outputs', 'nb16_reports')


class ReportAgentFull:
    """
    Agent 6 -- Full implementation.
    Replaces the stub in NB11.

    Generates:
    - Structured text report (BI-RADS/Lung-RADS/LI-RADS)
    - PDF report (ReportLab)
    - Patient-friendly plain language summary
    - Multi-language support
    - Saves to outputs/reports/<scan_id>/
    """

    def __init__(self, output_dir=None, language='en',
                 generate_pdf=True):
        self.output_dir   = output_dir or OUT
        self.language     = language
        self.generate_pdf = generate_pdf and REPORTLAB_AVAILABLE
        os.makedirs(self.output_dir, exist_ok=True)

    def generate(self, scan_data, gradcam_path=None):
        """
        Generate full report bundle for one scan.
        Returns dict with paths to all generated files + report text.
        """
        scan_id     = scan_data.get('scan_id', datetime.now().strftime('%Y%m%d%H%M%S'))
        cancer_type = scan_data.get('cancer_type', 'unknown')
        scan_dir    = os.path.join(self.output_dir, f'{scan_id}_{cancer_type}')
        os.makedirs(scan_dir, exist_ok=True)

        output_files = {}

        # 1. Structured text report
        report_text = generate_structured_report(scan_data)
        txt_path    = os.path.join(scan_dir, 'report.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        output_files['text_report'] = txt_path

        # 2. PDF report
        if self.generate_pdf:
            pdf_path = os.path.join(scan_dir, 'report.pdf')
            generate_pdf_report(scan_data, pdf_path, gradcam_path)
            output_files['pdf_report'] = pdf_path

        # 3. Patient summary
        patient_result = generate_patient_summary(scan_data, self.language)
        ps_path        = os.path.join(scan_dir, 'patient_summary.txt')
        with open(ps_path, 'w', encoding='utf-8') as f:
            f.write(patient_result['summary'])
        output_files['patient_summary'] = ps_path

        # 4. JSON metadata
        meta = {
            'scan_id'     : scan_id,
            'cancer_type' : cancer_type,
            'verdict'     : scan_data.get('verdict'),
            'cancer_prob' : scan_data.get('cancer_prob'),
            'tumor_type'  : scan_data.get('tumor_type'),
            'generated_at': datetime.now().isoformat(),
            'files'       : output_files,
        }
        meta_path = os.path.join(scan_dir, 'report_meta.json')
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)
        output_files['metadata'] = meta_path

        return {
            'report_text'   : report_text,
            'plain_summary' : patient_result['summary'],
            'output_files'  : output_files,
            'scan_dir'      : scan_dir,
        }


# Test full agent
report_agent = ReportAgentFull(generate_pdf=REPORTLAB_AVAILABLE)

print('Testing ReportAgentFull...')
gradcam_p = os.path.join(BASE, 'outputs', 'nb14_explainability', 'gradcam_results.png')

for scan in test_scans:
    result = report_agent.generate(
        scan,
        gradcam_path=gradcam_p if os.path.exists(gradcam_p) else None
    )
    print(f'  {scan["scan_id"]} ({scan["cancer_type"]}): {len(result["output_files"])} files')
    for name, path in result['output_files'].items():
        size = os.path.getsize(path) / 1024 if os.path.exists(path) else 0
        print(f'    {name:20s}: {os.path.basename(path)} ({size:.0f}KB)')

print('\nReportAgentFull OK')

Testing ReportAgentFull...
PDF saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports\48a808fc_brain\report.pdf (2021 KB)
  48a808fc (brain): 4 files
    text_report         : report.txt (1KB)
    pdf_report          : report.pdf (2021KB)
    patient_summary     : patient_summary.txt (0KB)
    metadata            : report_meta.json (1KB)
PDF saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports\20ce249f_breast\report.pdf (2021 KB)
  20ce249f (breast): 4 files
    text_report         : report.txt (1KB)
    pdf_report          : report.pdf (2021KB)
    patient_summary     : patient_summary.txt (0KB)
    metadata            : report_meta.json (1KB)
PDF saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\outputs\nb16_reports\841f187a_skin\report.pdf (2021 KB)
  841f187a (skin): 4 files
    text_report         : report.txt (1KB)
    pdf_report          : report.pdf (2021KB)
    patient_summary     : patient_summary.txt (0KB)
    meta

---
## Cell 6 - Summary

In [6]:
import os

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
OUT  = os.path.join(BASE, 'outputs', 'nb16_reports')

print('=' * 65)
print('  NOTEBOOK 16 - REPORT GENERATOR')
print('=' * 65)
print()
print('  Components built:')
print('    BIRADS / LUNGRADS / LIRADS  -- scoring tables')
print('    assign_birads/lungrads()    -- auto-score from model output')
print('    generate_structured_report()-- clinical text report')
print('    generate_pdf_report()       -- ReportLab PDF with verdict banner')
print('    generate_patient_summary()  -- 6th grade plain language')
print('    TRANSLATIONS                -- EN/ES/FR/HI/AR/ZH')
print('    ReportAgentFull             -- Full Agent 6 implementation')
print()
print('  Report bundle per scan (saved to outputs/nb16_reports/):')
print('    report.txt           -- structured clinical text')
print('    report.pdf           -- formatted PDF (if ReportLab installed)')
print('    patient_summary.txt  -- plain language for patient')
print('    report_meta.json     -- metadata + file paths')
print()
print('  Reports generated:')
n_reports = 0
for item in os.listdir(OUT):
    item_path = os.path.join(OUT, item)
    if os.path.isdir(item_path):
        files = os.listdir(item_path)
        print(f'    {item:35s}: {len(files)} files')
        n_reports += 1
print(f'  Total: {n_reports} scan reports')
print()
print('  Next: 17_Longitudinal_Tracker.ipynb')
print('    - Deformable registration')
print('    - Tumor growth rate cm3/month')
print('    - RECIST/RANO automation')
print('    - New lesion detection')
print('=' * 65)

  NOTEBOOK 16 - REPORT GENERATOR

  Components built:
    BIRADS / LUNGRADS / LIRADS  -- scoring tables
    assign_birads/lungrads()    -- auto-score from model output
    generate_structured_report()-- clinical text report
    generate_pdf_report()       -- ReportLab PDF with verdict banner
    generate_patient_summary()  -- 6th grade plain language
    TRANSLATIONS                -- EN/ES/FR/HI/AR/ZH
    ReportAgentFull             -- Full Agent 6 implementation

  Report bundle per scan (saved to outputs/nb16_reports/):
    report.txt           -- structured clinical text
    report.pdf           -- formatted PDF (if ReportLab installed)
    patient_summary.txt  -- plain language for patient
    report_meta.json     -- metadata + file paths

  Reports generated:
    20ce249f_breast                    : 4 files
    48a808fc_brain                     : 4 files
    841f187a_skin                      : 4 files
  Total: 3 scan reports

  Next: 17_Longitudinal_Tracker.ipynb
    - Deformab